# Per-Variant Surrogate Fit (CatBoost)

CatBoost is the selected surrogate architecture (justified in `04_surrogate_selection.ipynb` §7.6 — winner on L5B15 over Linear / GaussianNB / RandomForest / LightGBM). This notebook fits the *same* CatBoost configuration separately for each of the four offer variants and produces the artifacts consumed downstream (notebooks 06–08).

For each variant: load → drop absent/constant features → CatBoost-native encoding → stratified 80/20 split on propensity deciles → fit `train_catboost` defaults (500 iters, depth=6, lr=0.05, `random_seed=42`) → save full artifact set (`catboost_model.cbm`, `feature_cols.json`, `cat_cols.json`, `num_cols.json`, `train_idx.npy`, `test_idx.npy`, `fidelity.json`) → plot actual-vs-predicted + top-15 importances. The final section aggregates everything into the Results-chapter table and a 2×2 overview figure.

**Run `02_data_ingestion.ipynb` and `02b_data_ingestion_thirdparty.ipynb` first.**


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../src")
from my_project.features  import VARIANT_FEATURES
from my_project.metrics   import feature_ranking, fidelity_suite
from my_project.parsing   import THIRDPARTY_VARIANTS
from my_project.surrogate import build_feature_matrix, train_catboost

PROCESSED_DIR = Path("../data/processed")
ARTIFACT_ROOT = Path("../data/artifacts")

VARIANTS = ["L5B15", "CLUG", "BookingDotCom", "Cartrawler"]
print("Variants to fit:", VARIANTS)


## 1. Helper

`fit_variant(v)` does load → filter active → matrix → stratified split → fit → save full artifact set → return a dict with everything the per-variant plots and the §6 overview need.

The artifact filenames match what `06_explanation.ipynb` and onwards expect (`catboost_model.cbm`, `feature_cols.json`, …), so downstream notebooks consume these directly.


In [ ]:
# Per-variant fit + save. Results land in a single `artifacts` dict for §6.
artifacts: dict = {}

def fit_variant(variant: str) -> dict:
    cfg = VARIANT_FEATURES[variant]
    src = ("thirdparty_email_outbound.parquet"
           if variant in THIRDPARTY_VARIANTS
           else "luggage_email_outbound.parquet")

    df = pd.read_parquet(PROCESSED_DIR / src)
    df = df[df["pyName"] == variant].reset_index(drop=True)

    # Active predictors: present and non-constant in this variant's slice
    present = [f for f in cfg.features if f in df.columns]
    active  = [f for f in present if df[f].nunique(dropna=False) > 1]
    dropped = sorted(set(cfg.features) - set(active))

    X, y, cat_cols, num_cols = build_feature_matrix(df, active, cfg.numeric)

    bins = pd.qcut(y, q=10, labels=False, duplicates="drop")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=bins,
    )

    model    = train_catboost(X_train, y_train, cat_cols)
    y_pred   = model.predict(X_test)
    fidelity = fidelity_suite(y_test.values, y_pred)

    importances = pd.Series(
        model.get_feature_importance(),
        index=X_train.columns,
        name="importance",
    ).sort_values(ascending=False)

    # Save full artifact set used by notebooks 06–08
    art_dir = ARTIFACT_ROOT / variant
    art_dir.mkdir(parents=True, exist_ok=True)
    model.save_model(str(art_dir / "catboost_model.cbm"))
    (art_dir / "feature_cols.json").write_text(json.dumps(list(X.columns)))
    (art_dir / "cat_cols.json").write_text(json.dumps(cat_cols))
    (art_dir / "num_cols.json").write_text(json.dumps(num_cols))
    np.save(art_dir / "train_idx.npy", X_train.index.to_numpy())
    np.save(art_dir / "test_idx.npy",  X_test.index.to_numpy())
    (art_dir / "fidelity.json").write_text(
        json.dumps({**fidelity, "model": f"CatBoost ({variant})"}, indent=2)
    )

    result = {
        "variant":     variant,
        "n":           len(df),
        "n_train":     len(X_train),
        "n_test":      len(X_test),
        "n_features":  X.shape[1],
        "n_cat":       len(cat_cols),
        "n_num":       len(num_cols),
        "dropped":     dropped,
        "model":       model,
        "y_test":      y_test,
        "y_pred":      y_pred,
        "importances": importances,
        **fidelity,
    }
    artifacts[variant] = result
    return result


def _print_summary(r: dict) -> None:
    print(f"{r['variant']:<14} n={r['n']:,}  features={r['n_features']} "
          f"({r['n_cat']} cat, {r['n_num']} num)  "
          f"train/test={r['n_train']:,}/{r['n_test']:,}")
    if r["dropped"]:
        print(f"  dropped (absent/constant): {r['dropped']}")
    print(f"  R²={r['r2']:.4f}  RMSE={r['rmse']:.6f}  "
          f"ρ={r['spearman_rho']:.4f}  τ={r['kendall_tau']:.4f}  "
          f"KS={r['ks_stat']:.4f}")


def plot_variant(r: dict) -> None:
    """One row of figures per variant: actual-vs-pred (left) + top-15 importances (right)."""
    imp_top = r["importances"].head(15)[::-1]

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(13, max(4.5, len(imp_top) * 0.30)))

    # Actual vs predicted
    ax_l.scatter(r["y_test"], r["y_pred"], alpha=0.20, s=10, color="steelblue")
    lim = float(max(r["y_test"].max(), r["y_pred"].max())) * 1.05
    ax_l.plot([0, lim], [0, lim], "--", color="tomato", lw=1.2, label="y = x")
    ax_l.set_xlabel(r"Pega ADM propensity $\hat{p}$")
    ax_l.set_ylabel(r"Surrogate prediction $g(X)$")
    ax_l.set_title(f"{r['variant']} — actual vs predicted (R²={r['r2']:.3f}, ρ={r['spearman_rho']:.3f})",
                   fontsize=11)
    ax_l.legend(fontsize=9)

    # Top-15 importances
    ax_r.barh(imp_top.index, imp_top.values, color="steelblue", alpha=0.85)
    ax_r.set_xlabel("Mean decrease in RMSE")
    ax_r.set_title(f"{r['variant']} — top-15 feature importances", fontsize=11)
    ax_r.grid(axis="x", alpha=0.3, linestyle="--")
    ax_r.tick_params(axis="y", labelsize=8)

    plt.tight_layout()
    plt.show()


## 2. L5B15 — primary luggage variant
24 active predictors from the ADM snapshot.

In [ ]:
r_l5b15 = fit_variant("L5B15")
_print_summary(r_l5b15)
plot_variant(r_l5b15)


## 3. CLUG — generic luggage replication variant
21 active predictors. Distinct booking-context features vs L5B15.

In [ ]:
r_clug = fit_variant("CLUG")
_print_summary(r_clug)
plot_variant(r_clug)


## 4. BookingDotCom — hotel cross-sell (third-party)
36 active predictors — widest feature set, heavy interaction-history coverage.

In [ ]:
r_bookingdotcom = fit_variant("BookingDotCom")
_print_summary(r_bookingdotcom)
plot_variant(r_bookingdotcom)


## 5. Cartrawler — car-rental cross-sell (third-party)
25 active predictors. Includes `param::Param.FlightCost` and `param::Param.Age`, absent in luggage variants.

In [ ]:
r_cartrawler = fit_variant("Cartrawler")
_print_summary(r_cartrawler)
plot_variant(r_cartrawler)


## 6. Cross-variant overview

Single fidelity table for the Results chapter, plus a 2×2 actual-vs-predicted grid as a visual companion. Best per metric is highlighted (max for R²/ρ/τ, min for RMSE/KS).


In [ ]:
summary_df = (
    pd.DataFrame([artifacts[v] for v in VARIANTS])
    [["variant", "n", "r2", "rmse", "spearman_rho", "kendall_tau", "ks_stat"]]
    .rename(columns={
        "variant":      "Variant",
        "r2":           "R²",
        "rmse":         "RMSE",
        "spearman_rho": "Spearman ρ",
        "kendall_tau":  "Kendall τ",
        "ks_stat":      "KS statistic",
    })
)

display(
    summary_df.style
    .format({"n": "{:,}", "R²": "{:.4f}", "RMSE": "{:.6f}",
             "Spearman ρ": "{:.4f}", "Kendall τ": "{:.4f}", "KS statistic": "{:.4f}"})
    .highlight_max(subset=["R²", "Spearman ρ", "Kendall τ"], color="#d4edda")
    .highlight_min(subset=["RMSE", "KS statistic"],          color="#d4edda")
    .hide(axis="index")
)


In [ ]:
### 6.1  2×2 actual-vs-predicted grid (Results chapter figure)
fig, axes = plt.subplots(2, 2, figsize=(11, 9))

for ax, v in zip(axes.flat, VARIANTS):
    r = artifacts[v]
    ax.scatter(r["y_test"], r["y_pred"], alpha=0.20, s=8, color="steelblue")
    lim = float(max(r["y_test"].max(), r["y_pred"].max())) * 1.05
    ax.plot([0, lim], [0, lim], "--", color="tomato", lw=1.0)
    ax.set_xlabel(r"$\hat{p}$ (Pega ADM)", fontsize=10)
    ax.set_ylabel(r"$g(X)$ (surrogate)", fontsize=10)
    ax.set_title(f"{v}  —  R²={r['r2']:.3f},  ρ={r['spearman_rho']:.3f}",
                 fontsize=11)

plt.suptitle("CatBoost surrogate — actual vs predicted (held-out 20%)", fontsize=12, y=1.00)
plt.tight_layout()
plt.show()


In [ ]:
### 6.2  Top features per variant (shared overview)
# Rank-based view: for every feature that appears in any variant's top-10,
# show its rank under each variant (NaN = feature not active in that variant).
TOP_K = 10
top_per_variant = {v: artifacts[v]["importances"].head(TOP_K).index.tolist() for v in VARIANTS}
union = sorted({f for tops in top_per_variant.values() for f in tops})

rank_table = pd.DataFrame(index=union, columns=VARIANTS, dtype="float")
for v in VARIANTS:
    ranks = feature_ranking(artifacts[v]["importances"])  # 1-based
    rank_table[v] = ranks.reindex(union)

display(
    rank_table.sort_values(VARIANTS, na_position="last")
    .style
    .format(lambda x: "—" if pd.isna(x) else f"{int(x)}")
    .background_gradient(cmap="Greens_r", vmin=1, vmax=15, axis=None)
    .set_caption(f"Feature rank by importance (1 = most important). "
                 f"Union of each variant's top-{TOP_K}. Lower rank = darker green.")
)


In [ ]:
### 6.3  LaTeX table + persisted CSV
# pandas 3.x removed `booktabs=` from DataFrame.to_latex(); use Styler.to_latex(hrules=True),
# which emits \\toprule/\\midrule/\\bottomrule.
latex_table = (
    summary_df.style
    .format({"n": "{:,}", "R²": "{:.4f}", "RMSE": "{:.6f}",
             "Spearman ρ": "{:.4f}", "Kendall τ": "{:.4f}", "KS statistic": "{:.4f}"})
    .hide(axis="index")
    .to_latex(hrules=True)
)
print(latex_table)

out_csv = ARTIFACT_ROOT / "surrogate_fidelity_per_variant.csv"
summary_df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
